In [1]:
import numpy as np, pandas as pd, time
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

d = np.load("../data/processed/fd001_windows.npz")
X_tr, y_tr, X_val, y_val = d["X_tr"], d["y_tr"], d["X_val"], d["y_val"]
X_test, y_test = d["X_test"], d["y_test"]
features = list(d["features"])
y_test = np.minimum(y_test, 125)   # same cap as training, for a fair comparison
y_test_raw = d["y_test"]   # uncapped true RUL

def rmse(y, p):
    return float(np.sqrt(np.mean((y - p) ** 2)))

def nasa_score(y, p):
    e = p - y   # late (e>0) is penalised more than early
    return float(np.sum(np.where(e < 0, np.exp(-e / 13) - 1, np.exp(e / 10) - 1)))

results = {}

In [2]:
def summarize(X):
    t = np.arange(X.shape[1])
    slope = ((X - X.mean(1, keepdims=True)) * (t - t.mean())[None, :, None]).sum(1) / ((t - t.mean()) ** 2).sum()
    return np.concatenate([X[:, -1], X.mean(1), X.std(1), X.min(1), X.max(1), slope], axis=1)

F_tr, F_val, F_test = summarize(X_tr), summarize(X_val), summarize(X_test)
print(F_tr.shape)

(14241, 90)


In [3]:
t0 = time.time()
rf = RandomForestRegressor(n_estimators=300, min_samples_leaf=3, n_jobs=-1, random_state=42)
rf.fit(F_tr, y_tr)
results["RandomForest"] = {
    "val_RMSE": rmse(y_val, rf.predict(F_val)),
    "test_RMSE": rmse(y_test, rf.predict(F_test)),
    "test_NASA": nasa_score(y_test, rf.predict(F_test)),
    "time_s": time.time() - t0,
}
print(results["RandomForest"])

{'val_RMSE': 12.876663208447079, 'test_RMSE': 12.91076891003451, 'test_NASA': 278.2659506787695, 'time_s': 15.073121070861816}


In [4]:
t0 = time.time()
xgb = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                   subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=42)
xgb.fit(F_tr, y_tr)
results["XGBoost"] = {
    "val_RMSE": rmse(y_val, xgb.predict(F_val)),
    "test_RMSE": rmse(y_test, xgb.predict(F_test)),
    "test_NASA": nasa_score(y_test, xgb.predict(F_test)),
    "time_s": time.time() - t0,
}
print(results["XGBoost"])
pd.DataFrame(results).T

{'val_RMSE': 12.486480712890625, 'test_RMSE': 12.976818084716797, 'test_NASA': 265.4316711425781, 'time_s': 1.3992362022399902}


,val_RMSE,test_RMSE,test_NASA,time_s
RandomForest,12.876663,12.910769,278.265951,15.073121
XGBoost,12.486481,12.976818,265.431671,1.399236


In [5]:
for name, m in [("RandomForest", rf), ("XGBoost", xgb)]:
    p = m.predict(F_test)
    results[name]["test_RMSE_raw"] = rmse(y_test_raw, p)
    results[name]["test_NASA_raw"] = nasa_score(y_test_raw, p)
pd.DataFrame(results).T.to_csv("../results/baselines.csv")
pd.DataFrame(results).T.round(2)

,val_RMSE,test_RMSE,test_NASA,time_s,test_RMSE_raw,test_NASA_raw
RandomForest,12.88,12.91,278.27,15.07,14.11,310.99
XGBoost,12.49,12.98,265.43,1.40,14.09,295.75


In [6]:
rows = []
for seed in [42, 1, 2, 3, 4]:
    models = [
        ("RandomForest", RandomForestRegressor(n_estimators=300, min_samples_leaf=3, n_jobs=-1, random_state=seed)),
        ("XGBoost", XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                                 subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=seed)),
    ]
    for name, m in models:
        m.fit(F_tr, y_tr)
        p = m.predict(F_test)
        rows.append({"model": name, "seed": seed, "val_RMSE": rmse(y_val, m.predict(F_val)),
                     "test_RMSE_raw": rmse(y_test_raw, p), "test_NASA_raw": nasa_score(y_test_raw, p)})
    print("seed", seed, "done")

tree_runs = pd.DataFrame(rows)
tree_runs.to_csv("../results/tree_runs.csv", index=False)
tree_runs.groupby("model")[["val_RMSE", "test_RMSE_raw", "test_NASA_raw"]].agg(["mean", "std"]).round(2)

seed 42 done


seed 1 done


seed 2 done


seed 3 done


seed 4 done


val_RMSE       test_RMSE_raw       test_NASA_raw       
                 mean   std          mean   std          mean    std
model                                                               
RandomForest    12.82  0.04         14.19  0.07        325.47   9.97
XGBoost         12.42  0.06         13.89  0.22        286.31  10.20